# YOLO 검출 결과 시각화

`outputs/yolo_YYYYMMDD_HHMMSS/predictions.json` 기반으로 bbox와 class를 이미지에 표시합니다.

**NumPy 2.x 오류 시**: `pip install "numpy<2"` 또는 `rtdetr310` 등 호환 conda env의 커널을 선택하세요.

In [2]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# YOLO 출력 디렉토리 (수정 가능)
OUTPUT_DIR = Path("outputs/yolo_20260309_162537")

# NumPy 2.x 호환 문제 시: pip install "numpy<2" 또는 rtdetr310 등 다른 env 사용


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/vailab02/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/vailab02/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/vailab02/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/home

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
# predictions.json, image_paths.json 로드
with open(OUTPUT_DIR / "predictions.json") as f:
    predictions = json.load(f)

with open(OUTPUT_DIR / "image_paths.json") as f:
    image_paths = json.load(f)

print(f"샘플 수: {len(predictions)}")
print("샘플 ID:", list(predictions.keys())[:5], "..." if len(predictions) > 5 else "")

In [ ]:
def get_class_color(class_name: str) -> str:
    """클래스별 고정 색상 (재현 가능)"""
    colors = {
        "person": "lime",
        "car": "cyan",
        "tv": "orange",
        "bottle": "yellow",
        "chair": "magenta",
        "light": "gold",
    }
    return colors.get(class_name, "white")


def visualize_yolo_detections(
    image_path: str,
    detections: list,
    ax=None,
    title: str = "",
    show_conf: bool = True,
):
    """
    YOLO 검출 결과를 이미지에 그립니다.
    detections: [{"bbox": [x,y,w,h], "class": str, "conf": float}, ...]
    """
    img = Image.open(image_path).convert("RGB")

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)  # PIL Image 직접 사용 (numpy 불필요)
    ax.set_title(title or Path(image_path).name, fontsize=10)

    for i, det in enumerate(detections):
        x, y, w, h = det["bbox"]
        cls = det["class"]
        conf = det.get("conf", 0)
        color = get_class_color(cls)

        rect = patches.Rectangle((x, y + h), w, -h, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)

        label = cls
        if show_conf:
            label += f" {conf:.2f}"
        ax.text(
            x, max(0, y - 4),
            label,
            color=color,
            fontsize=7,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6),
        )

    ax.axis("off")
    if ax is None:
        plt.tight_layout()

In [ ]:
# 모든 샘플 시각화 (그리드)
n = len(predictions)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1 and cols == 1:
    axes = [[axes]]
elif rows == 1:
    axes = [axes]

for idx, (sample_id, dets) in enumerate(predictions.items()):
    row, col = idx // cols, idx % cols
    ax = axes[row, col]
    img_path = image_paths.get(sample_id)
    if not img_path or not Path(img_path).exists():
        ax.text(0.5, 0.5, f"이미지 없음:\n{img_path or sample_id}", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        continue
    visualize_yolo_detections(img_path, dets if dets else [], ax=ax, title=sample_id)

# 빈 subplot 숨기기
for idx in range(n, rows * cols):
    row, col = idx // cols, idx % cols
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 특정 샘플만 개별 시각화 (인덱스 또는 ID로 선택)
sample_ids = list(predictions.keys())[:2]  # 처음 2개

fig, axes = plt.subplots(1, len(sample_ids), figsize=(8 * len(sample_ids), 8))
if len(sample_ids) == 1:
    axes = [axes]

for ax, sample_id in zip(axes, sample_ids):
    img_path = image_paths.get(sample_id)
    dets = predictions.get(sample_id, [])
    if img_path and Path(img_path).exists():
        visualize_yolo_detections(img_path, dets, ax=ax, title=f"{sample_id} ({len(dets)} detections)")
    else:
        ax.text(0.5, 0.5, f"Not found: {img_path}", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

plt.tight_layout()
plt.show()